# 06 — Alters-Kohorten & Spieler-Tabelle

**Frage:** Wo kommen die Rating-Zuwächse her? Treiben jüngere oder ältere Spieler*innen die Verbesserung?

1. Kohortenbildung nach Alter **in 2015** (erste Periode): <20, 20–30, 30–40, 40–50, >50
2. Pro Kohorte × Gruppe: Mean/Median der kumulativen Rating-Änderung
3. Heatmap Kohorte × Jahr
4. **Spieler-Tabelle** mit Name, aktueller ELO, Gesamt-Δ, Δ pro Jahr (2015–2025)

Filter: `active = TRUE AND analysis_group IS NOT NULL`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _setup import load_query, apply_style, GROUP_PALETTE, GROUP_ORDER

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

apply_style()
pd.set_option('display.max_rows', 250)

## Datenbasis laden

In [ ]:
sql = '''
SELECT
    gr.fide_id,
    p.name,
    p.analysis_group,
    p.std_rating AS current_rating,
    p.birth_year,
    EXTRACT(YEAR FROM gr.period)::int AS year,
    gr.rating_change_weighted
FROM game_results gr
JOIN players p ON p.fide_id = gr.fide_id
WHERE p.active = TRUE AND p.analysis_group IS NOT NULL
'''
df = load_query(sql)
df['rating_change_weighted'] = df['rating_change_weighted'].astype(float)
df['age_2015'] = 2015 - df['birth_year']

def age_bucket(a):
    if pd.isna(a): return 'unknown'
    if a < 20: return '<20'
    if a < 30: return '20-30'
    if a < 40: return '30-40'
    if a < 50: return '40-50'
    return '>50'
df['cohort'] = df['age_2015'].apply(age_bucket)
COHORT_ORDER = ['<20', '20-30', '30-40', '40-50', '>50']
print('Partien:', len(df), '| Spieler:', df['fide_id'].nunique())
df[['fide_id','name','analysis_group','birth_year','age_2015','cohort']].drop_duplicates('fide_id').head()

## 1. Kohorten-Besetzung

Wie viele Spieler pro Kohorte × Gruppe?

In [ ]:
players_meta = df[['fide_id','analysis_group','cohort']].drop_duplicates('fide_id')
cohort_counts = (
    players_meta.groupby(['analysis_group','cohort']).size()
    .unstack('cohort').reindex(columns=COHORT_ORDER, fill_value=0)
)
cohort_counts

## 2. Kumulativer Rating-Gewinn pro Kohorte

Pro Spieler Σ `rating_change_weighted` über 2015–2025, dann Mean/Median je Kohorte × Gruppe.

In [ ]:
total_per_player = (
    df.groupby(['analysis_group','cohort','fide_id'])['rating_change_weighted']
      .sum().reset_index(name='total_sum')
)
cohort_stats = total_per_player.groupby(['analysis_group','cohort']).agg(
    n_players=('fide_id','nunique'),
    mean_total=('total_sum','mean'),
    median_total=('total_sum','median'),
).round(1).reset_index()
cohort_stats.pivot(index='cohort', columns='analysis_group', values='mean_total').reindex(COHORT_ORDER)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=total_per_player, x='cohort', y='total_sum',
    hue='analysis_group', hue_order=GROUP_ORDER, palette=GROUP_PALETTE,
    order=COHORT_ORDER, ax=ax,
)
sns.stripplot(
    data=total_per_player, x='cohort', y='total_sum',
    hue='analysis_group', hue_order=GROUP_ORDER, dodge=True,
    order=COHORT_ORDER, color='black', alpha=0.3, size=3, ax=ax,
    legend=False,
)
ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.set_ylabel('Σ rating_change_weighted (2015–2025)')
ax.set_xlabel('Alter in 2015')
ax.set_title('Kumulativer Rating-Gewinn pro Alterskohorte')
plt.tight_layout(); plt.show()

## 3. Heatmap Kohorte × Jahr

Mean Jahres-Summe pro Kohorte × Jahr, getrennt nach Gruppe. Zeigt, ob Gewinne gleichmäßig verteilt sind oder einzelne Jahre/Kohorten dominieren.

In [ ]:
yearly_per_player = (
    df.groupby(['analysis_group','cohort','fide_id','year'])['rating_change_weighted']
      .sum().reset_index(name='yearly_sum')
)
yearly_cohort = (
    yearly_per_player.groupby(['analysis_group','cohort','year'])['yearly_sum']
                     .mean().round(1).reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, grp in zip(axes, GROUP_ORDER):
    sub = (yearly_cohort[yearly_cohort['analysis_group']==grp]
           .pivot(index='cohort', columns='year', values='yearly_sum')
           .reindex(COHORT_ORDER))
    sns.heatmap(sub, annot=True, fmt='.1f', cmap='RdYlGn', center=0, ax=ax,
                cbar_kws={'label':'Ø Δ pro Spieler'})
    ax.set_title(grp)
    ax.set_xlabel('Jahr'); ax.set_ylabel('Kohorte')
plt.tight_layout(); plt.show()

## 4. Spieler-Tabelle

Pro Spieler: aktueller ELO, Gesamt-Δ, Jahres-Δ 2015–2025. Sortiert alphabetisch.

In [ ]:
yearly_wide = (
    yearly_per_player.pivot_table(
        index='fide_id', columns='year', values='yearly_sum', fill_value=0
    ).round(1)
)
yearly_wide.columns = [f'Δ {int(c)}' for c in yearly_wide.columns]

meta = df[['fide_id','name','analysis_group','current_rating','age_2015','cohort']].drop_duplicates('fide_id').set_index('fide_id')
totals = total_per_player.set_index('fide_id')['total_sum'].round(1).rename('Σ 2015–2025')

table = meta.join(totals).join(yearly_wide)
table = table.rename(columns={
    'name':'Name','analysis_group':'Gruppe',
    'current_rating':'ELO aktuell','age_2015':'Alter 2015','cohort':'Kohorte',
})
table = table.sort_values('Name').reset_index(drop=True)
print(f'{len(table)} Spieler')
table

### CSV-Export

In [ ]:
out = Path('player_rating_changes.csv')
table.to_csv(out, index=False)
print(f'wrote {out}')